In [ ]:
# Script for styling the Jupyter Notebook using an external CSS file
from IPython.display import HTML

with open("style.css", "r") as f:
    css = f.read()

HTML(f"<style>{css}</style>")

# Graph Neural Network proof of concept

This notebook generates the proof of concept for GNNs. We generated a control data using a column boosting method and trained a GNN model with a GraphConv architecture. We evaluated the following:

1. **Extinction quantification:** Tested whether the level at which extinction is measured 
   affects predictive performance — specifically, whether using full-community extinction 
   metrics versus sub-community metrics (composed only of surviving species) is sufficient 
   for prediction.
2. **Model accuracy:** Assessed whether the model can predict the target variable at a 
   meaningful level.
3. **Input features:** Evaluated model performance when incorporating node-level statistics 
   compared to using dummy input features only.

Results path: `/home/mriveraceron/glv-research/Results/gnn_proof`

Training data paths: 
- `/home/mriveraceron/glv-research/data_tensors/gnn_proof_full`
-  `/home/mriveraceron/glv-research/data_tensors/gnn_proof_sub`

Model performance results are shown below.

# Extinction quantification

In [ ]:
# Plots for quantification
import pandas as pd
from IPython.display import display

dir = '/home/mriveraceron/glv-research/Results/gnn_proof'
df = pd.read_csv(f'{dir}/result_table.csv')
display(df.sort_values(by="pearson_corr", ascending=False))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import glob
from scipy.stats import pearsonr, spearmanr
import os

def corr_plt(data, method, ax, label):
    # --- Config ---
    LIMS        = [-0.02, 1.05]
    TICK_MAJOR  = 0.2
    TICK_MINOR  = 0.05
    FMT         = '%.1f'
    # --- Data & correlations ---
    mt, mp = data['mt'], data['mp']
    r_p, _ = pearsonr(mt, mp)
    r_s, _ = spearmanr(mt, mp)
    x, y   = np.log1p(mp), np.log1p(mt)
    # --- Figure ---
    # Hexbin
    hb = ax.hexbin(x, y, gridsize=45, cmap='YlOrRd', mincnt=1, bins='log', linewidths=0.15, edgecolors='none')
    # Colorbar
    cb = fig.colorbar(hb, ax=ax, pad=0.03, shrink=0.82, aspect=22)
    cb.set_label('Recuento (escala log)', fontsize=11, labelpad=8)
    cb.ax.tick_params(labelsize=10, direction='in')
    cb.outline.set_linewidth(0.8)
    # Identity line
    ax.plot(LIMS, LIMS, color='0.3', linewidth=1.2, linestyle='--', alpha=0.85, zorder=5, label='Correlación perfecta')
    # Labels & titles
    ax.set_xlabel('Keystoneness predicho', labelpad=8)
    ax.set_ylabel('Keystoneness observado', labelpad=8)
    ax.set_xlim(LIMS)
    ax.set_ylim(LIMS)
    fig.suptitle('GraphConv — valores de keystoneness en datos con incremento de columna', fontsize=15, fontfamily='serif')
    ax.set_title(method, fontsize=11, color='0.4', pad=6)
    # Ticks (both axes)
    for axis in (ax.xaxis, ax.yaxis):
        axis.set_major_locator(ticker.MultipleLocator(TICK_MAJOR))
        axis.set_minor_locator(ticker.MultipleLocator(TICK_MINOR))
        axis.set_major_formatter(ticker.FormatStrFormatter(FMT))
    # Correlation annotation
    ax.text(
        0.05, 0.95,
        f'$r$ = {r_p.item():.3f}$\;$(Pearson)\n$\\rho$ = {r_s.item():.3f}$\;$(Spearman)',
        transform=ax.transAxes, ha='left', va='top',
        fontsize=10, fontfamily='serif', zorder=10,
        bbox=dict(boxstyle='round,pad=0.5,rounding_size=0.4',facecolor='white', edgecolor='0.75', linewidth=0.8, alpha=0.9),
    )
    # Panel label
    ax.text(-0.1, 1.05, label, transform=ax.transAxes, fontsize=14, fontweight='bold', va='top', ha='left')
    # Legend & spines
    ax.legend(loc='lower right', fontsize=10, framealpha=0.9, edgecolor='0.75')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    return fig

# Script chunk for models performance
dir = '/home/mriveraceron/glv-research/Results/gnn_proof'
paths = sorted(glob.glob(f'{dir}/**/*.npz'))  # sorted for consistent ordering
plot_titles = [
    'Comunidad completa · sin estadísticas de nodo',
    'Comunidad completa · con estadísticas de nodo',
    'Subcomunidad · sin estadísticas de nodo',
    'Subcomunidad · con estadísticas de nodo',
]
# Generate panel plot
plt.clf()
plt.close('all')
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(15, 10))
plot_id = ['A', 'B', 'C', 'D']

data = np.load(paths[0])
for data_path, method, ax, label in zip(paths, plot_titles, axes.flat, plot_id):
    # print(f'plotting {data_path} with {method}')
    data = np.load(data_path)
    p = corr_plt(data, method, ax, label)

save_path = f'{dir}/preds_panel.png'
plt.savefig(save_path)
plt.show()
print(f'>> Image saved at: {save_path}')

## Interpretation

Based on the figures above, there is no clear distinction between measuring extinction at the full-community level versus the subcommunity level. This may be attributed to the proportional increase in relative abundance within the subcommunity following extinction.

Notably, models using dummy features (A and C) outperformed those incorporating node statistics (B and D). This may be explained by the fact that the control dataset follows a distinct pattern in its interaction matrix, suggesting that the adjacency structure and edge weights alone are sufficient for learning keystoneness. The additional incorporation of node statistics may introduce redundant parameters, hindering rather than improving the learning process.

# Confussion matrixes are shown below


In [ ]:
# Confussion matrix script

# Script chunk for models performance
dir = '/home/mriveraceron/glv-research/Results/gnn_proof'
paths = sorted(glob.glob(f'{dir}/**/*.npz'))  # sorted for consistent ordering
exeriment = [
    'Full community dummy',
    'Full community feats',
    'Subcommunity dummy',
    'Subcommunity feats',
]

for model, path in zip(exeriment, paths):
    data        = np.load(path)
    idxt        = data['idxt']
    idxp        = data['idxp']
    graph_nodes = data['nodes']
    # Compute mask once, reuse for tp and fp
    correct = idxt == idxp
    tp = correct.sum()
    fp = len(idxt) - tp          # avoids a second full pass over the array
    fn = fp
    tn = graph_nodes.sum() - len(graph_nodes) - fp   # vectorized sum(nodes - 1)
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    ppv      = tp / (tp + fp)
    print(f'Model {model} | Accuracy: {accuracy:.4f} | PPV: {ppv:.4f} | By chance: {1/graph_nodes.mean():.4f}')
    df_cm = pd.DataFrame(
        [[tp, fp], [fn, tn]],
        index   = ['Expected_Positive', 'Expected_Negative'],
        columns = ['Predicted_Positive', 'Predicted_Negative']
    )
    display(df_cm)
    print('\n')